# EE2211 Tutorial 11 — K-means Clustering

Covers Questions 5, 6, 7 (the computational questions).

**Key concepts**:
- Naive K-means: random init → assign → update → repeat until convergence
- Local optima: different initializations can give different results; run multiple times
- K-means is **unsupervised** — no class labels used during clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## Question 5 — Manual K-means Trace (8 data points, K=2)

Points: x1=(0,0), x2=(0,1), x3=(1,1), x4=(1,0), x5=(3,0), x6=(3,1), x7=(4,0), x8=(4,1)

Initial centroids: c1=(0,0), c2=(3,0)

Expected result: c1=(0.5, 0.5), c2=(**blank1**, **blank2**)

In [ ]:
X = np.array([[0,0],[0,1],[1,1],[1,0],[3,0],[3,1],[4,0],[4,1]], dtype=float)
centroids = np.array([[0,0],[3,0]], dtype=float)

def assign(X, centroids):
    dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)  # (N, K)
    return np.argmin(dists, axis=1)

def update(X, labels, K):
    return np.array([X[labels == k].mean(axis=0) for k in range(K)])

for i in range(20):
    labels = assign(X, centroids)
    new_c = update(X, labels, K=2)
    print(f'Iter {i+1}: assignments={labels.tolist()}')
    print(f'         c1={new_c[0]}  c2={new_c[1]}')
    if np.allclose(centroids, new_c):
        print('\nConverged!')
        break
    centroids = new_c

print(f'\nAnswer: c2 = ({centroids[1][0]:.1f}, {centroids[1][1]:.1f})')

## Question 6 — Naive K-means on Generated 2D Data

Generate 600 points (200 per cluster) centred at (2,2), (4,4), (6,1).
Implement naive K-means and visualise with K=3 then K=5.

In [ ]:
np.random.seed(42)
center_1, center_2, center_3 = np.array([2,2]), np.array([4,4]), np.array([6,1])
data_1 = np.random.randn(200, 2) + center_1
data_2 = np.random.randn(200, 2) + center_2
data_3 = np.random.randn(200, 2) + center_3
data = np.concatenate([data_1, data_2, data_3], axis=0)

plt.figure(figsize=(5, 4))
plt.scatter(data[:, 0], data[:, 1], s=7, alpha=0.5)
plt.title('Raw data (no labels)')
plt.tight_layout(); plt.show()

In [ ]:
def naive_kmeans(X, K, seed=42, max_iters=200):
    """Naive K-means: random centroid init from data points."""
    rng = np.random.RandomState(seed)
    centroids = X[rng.choice(len(X), K, replace=False)].copy()
    for _ in range(max_iters):
        labels = assign(X, centroids)
        new_c  = update(X, labels, K)
        if np.allclose(centroids, new_c):
            break
        centroids = new_c
    return labels, centroids

COLORS = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange', 'tab:purple']

def plot_kmeans(data, labels, centroids, K, title):
    plt.figure(figsize=(6, 5))
    for k in range(K):
        mask = labels == k
        plt.scatter(data[mask, 0], data[mask, 1], s=7, c=COLORS[k], alpha=0.6, label=f'Cluster {k+1}')
        plt.scatter(centroids[k, 0], centroids[k, 1], s=250, c=COLORS[k], marker='*',
                    edgecolors='black', linewidths=1.5, zorder=5)
    plt.title(title); plt.legend(markerscale=3); plt.tight_layout(); plt.show()

# (i) K = 3
labels3, centroids3 = naive_kmeans(data, K=3)
print('Centroids (K=3):\n', np.round(centroids3, 3))
plot_kmeans(data, labels3, centroids3, K=3, title='K-means K=3')

In [ ]:
# (ii) K = 5
labels5, centroids5 = naive_kmeans(data, K=5)
print('Centroids (K=5):\n', np.round(centroids5, 3))
plot_kmeans(data, labels5, centroids5, K=5, title='K-means K=5')

## Question 7 — K-means on Iris (K=3), Compare with True Labels

Apply naive K-means to all 150 iris samples (4D features, labels hidden).
Then align cluster IDs to class labels and measure accuracy.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_true = iris.data, iris.target

labels_iris, centroids_iris = naive_kmeans(X_iris, K=3, seed=42)

# Align cluster IDs → class labels (majority vote per cluster)
mapping = {k: Counter(y_true[labels_iris == k]).most_common(1)[0][0] for k in range(3)}
y_pred = np.array([mapping[l] for l in labels_iris])

accuracy = np.mean(y_pred == y_true)
print(f'Cluster → class mapping: {mapping}')
print(f'K-means accuracy vs true labels: {accuracy:.3f} ({accuracy*100:.1f}%)')
print('Note: class 0 (setosa) is linearly separable; classes 1 and 2 overlap → some errors expected.')

# Scatter (first 2 features for 2D visualisation)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k in range(3):
    mask = labels_iris == k
    axes[0].scatter(X_iris[mask, 0], X_iris[mask, 1], s=20, c=COLORS[k], alpha=0.7, label=f'Cluster {k}')
axes[0].set_title('K-means clusters (features 0 vs 1)'); axes[0].legend()
for c in range(3):
    mask = y_true == c
    axes[1].scatter(X_iris[mask, 0], X_iris[mask, 1], s=20, c=COLORS[c], alpha=0.7, label=iris.target_names[c])
axes[1].set_title('True labels'); axes[1].legend()
plt.tight_layout(); plt.show()